In [9]:
# Load JSON 

import json 
import pandas as pd

with open("example_italian_science_dataset.json") as f:
    data = json.load(f)

# Convert JSON into dataframe
df = pd.DataFrame.from_dict(data, orient="index")

# Move OMIDs into a column 
df.reset_index(inplace=True)
df.rename(columns={"index": "omid"}, inplace=True)

print(df.head())

                  omid                            pids  \
0  omid:br/06000000001  [doi:10.1145/example.2026.001]   
1  omid:br/06000000002  [doi:10.1145/example.2026.002]   
2  omid:br/06000000003  [doi:10.1145/example.2026.003]   
3  omid:br/06000000004  [doi:10.1145/example.2026.004]   
4  omid:br/06000000005  [doi:10.1145/example.2026.005]   

                                       organizations                countries  \
0       [Stanford University, University of Bologna]   [United States, Italy]   
1        [University of Oxford, University of Milan]  [United Kingdom, Italy]   
2         [Sorbonne University, University of Turin]          [France, Italy]   
3                  [ETH Zurich, University of Padua]     [Switzerland, Italy]   
4  [Massachusetts Institute of Technology, Univer...   [United States, Italy]   

  direction  
0   inbound  
1  outbound  
2   inbound  
3  outbound  
4   inbound  


# Count countries by direction


In [ ]:


df = df.explode(["organizations", "countries"])

# Remove Italian institutions 
italian_universities = [
    "University of Bologna",
    "University of Milan",
    "University of Turin",
    "University of Padua",
    "University of Eastern Piedmont",
    "Scuola Normale Superiore"
]

df = df[~df["organizations"].isin(italian_universities)]

print(df.head())

# Count

country_counts = (
    df.groupby(["countries", "direction"])
        .size()
        .reset_index(name="count")
)

print(country_counts)

                  omid                            pids  \
0  omid:br/06000000001  [doi:10.1145/example.2026.001]   
1  omid:br/06000000002  [doi:10.1145/example.2026.002]   
2  omid:br/06000000003  [doi:10.1145/example.2026.003]   
3  omid:br/06000000004  [doi:10.1145/example.2026.004]   
4  omid:br/06000000005  [doi:10.1145/example.2026.005]   

                           organizations       countries direction  
0                    Stanford University   United States   inbound  
1                   University of Oxford  United Kingdom  outbound  
2                    Sorbonne University          France   inbound  
3                             ETH Zurich     Switzerland  outbound  
4  Massachusetts Institute of Technology   United States   inbound  
         countries direction  count
0        Australia  outbound      2
1          Belgium  outbound      2
2           Canada   inbound      2
3          Denmark  outbound      2
4          Finland  outbound      2
5           France   

In [17]:
country_counts["value"] = country_counts.apply(
    lambda row: row["count"] if row["direction"] == "outbound"
    else -row["count"],
    axis=1
)

country_counts = country_counts.sort_values("value")

print(country_counts)

         countries direction  count  value
14   United States   inbound      8     -8
2           Canada   inbound      2     -2
5           France   inbound      2     -2
7            Japan   inbound      2     -2
8      Netherlands   inbound      2     -2
10     South Korea   inbound      2     -2
12  United Kingdom   inbound      2     -2
0        Australia  outbound      2      2
1          Belgium  outbound      2      2
3          Denmark  outbound      2      2
4          Finland  outbound      2      2
9        Singapore  outbound      2      2
11     Switzerland  outbound      2      2
13  United Kingdom  outbound      2      2
15   United States  outbound      2      2
6          Germany  outbound      4      4


# Plot the diverging bars 

In [40]:
import plotly.express as px

# create signed values
country_counts["value"] = country_counts.apply(
    lambda row: row["count"] if row["direction"] == "outbound"
    else -row["count"],
    axis=1
)

# sort for cleaner visualization
country_counts = country_counts.sort_values("value")

# create chart
fig = px.bar(
    country_counts,
    x="value",
    y="countries",
    color="direction",
    orientation="h",

    custom_data = ["count", "direction"],

    color_discrete_map={
        "inbound": "#c44e52",
        "outbound": "#4c72b0"
    },

    hover_data={
        "value": False,
        "count": True
    },

    title="Geographical Distribution of Citation Relationships<br><sup>University of Bologna</sup>"
)

# add central line
fig.add_vline(x=0, line_width=1.5, line_color="gray")

# improve layout
fig.update_layout(
    template="plotly_white",
    height=700,
    bargap=0.15,
    legend_title_text="",

    xaxis_title="Citation Count",
    yaxis_title="",

    font=dict(
        size=14
    )   
)

# create balance
max_value = country_counts["value"].abs().max()

fig.update_xaxes(
    range=[-max_value - 1, max_value + 1],
    showgrid=False
)

fig.update_yaxes(
    showgrid=False
)

# hover labels 
fig.update_traces(
    hovertemplate=
    "<b>%{y}</b><br>" +
    "Citations: %{customdata[0]}<br>" +
    "Direction: %{customdata[1]}<extra></extra>"
)

fig.show()

Potential issues:
* pay attention to duplicates for organizations and countries
* pay attention to countries normalization (USA, Unites States, U.S)
* should we filter out Italy? Always? 
* consider using top 15 countries if they are too much or only countries with >= 5 citations